In [5]:
# %pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 16.3 MB/s eta 0:00:00


In [6]:
import warnings
from functools import partial

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFECV
from sklearn.experimental import enable_halving_search_cv
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from sklearn.base import TransformerMixin, RegressorMixin, BaseEstimator
from sklearn.model_selection import TimeSeriesSplit, HalvingRandomSearchCV, train_test_split, GridSearchCV, ParameterGrid
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, make_scorer
from xgboost import XGBRegressor

import lightgbm as lgb
import optuna

import requests
from functools import reduce

from scipy.stats import loguniform, randint, uniform


pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [2]:
def MirrorLog(X, c:float=(1/3)):

    '''
    X   :   The data that will be mirror-log transformed.
    c   :   Constant parameter. Default is 1/3, as suggested in the literature.
    '''

    X = np.asarray(X)
    X_new = np.empty_like(X)
    X_new[X != 0] = np.sign(X[X != 0]) * (np.log(np.abs(X[X != 0]) * (1/c)) + np.log(c))
    X_new[X == 0] = 0
    return X_new

# reverses the mirror log transformation, restoring data—including negative or zero values—back to its original scale by undoing the signed logarithmic mapping.
def InverseMirrorLog(X, c:float=(1/3)):

    '''
    X   :   The data that will be inverse mirror-log transformed.
    c   :   Constant parameter. Default is 1/3, as suggested in the literature.
    '''

    X = np.asarray(X)
    X_inv = np.empty_like(X)
    X_inv[X != 0] = np.sign(X[X != 0]) * (np.exp(np.abs(X[X != 0]) - np.log(c)) - (1/c))
    X_inv[X == 0] = 0
    return X_inv


class MirrorLogNormScaler(TransformerMixin, BaseEstimator):

    def __init__(self, mirrorlog_kwargs:dict=None, normalizer=None):

        '''
        mirrorlog_kwargs    :   Keyword arguments for the mirror-log transformation. Default is {'c': 1/3}.
        normalizer          :   Normalizer to use after mirror-log transformation. Default is MinMaxScaler.
        '''

        self.mirrorlog_kwargs = mirrorlog_kwargs
        self.normalizer = normalizer
        _normalizer = self.normalizer or MinMaxScaler()
        self._normalizer = _normalizer

        mirrorlog_kwargs_ = self.mirrorlog_kwargs or {'c': 1/3}

        self._mlog_scaler = FunctionTransformer(
            func=partial(MirrorLog, **mirrorlog_kwargs_),
            inverse_func=partial(InverseMirrorLog, **mirrorlog_kwargs_),
            check_inverse=False
        )

    def fit(self, X, y=None):

        '''
            X   :   The data that will be mirror-log transformed then used to compute the per-feature minimum and maximum used for later scaling along the features axis.
            y   :   Ignored.
        '''

        X_mlog = self._mlog_scaler.fit_transform(X)
        self._normalizer.fit(X_mlog)
        return self

    def transform(self, X):

        '''
        X   :   The data that will be mirror-log transformed then min-max scaled.
        '''

        X_mlog = self._mlog_scaler.transform(X)
        X_new = self._normalizer.transform(X_mlog)
        return X_new

    def inverse_transform(self, X):

        '''
        X   :   The data that will be inverse min-max scaled then inverse mirror-log transformed.
        '''

        X_mlog = self._normalizer.inverse_transform(X)
        X_inv = self._mlog_scaler.inverse_transform(X_mlog)
        return X_inv


class ProcessingPipeline(RegressorMixin, BaseEstimator):

    '''
    rfecv_kwargs        :   Keyword arguments for the RFECV feature selection step. Estimator for feature importance must be provided. Default is a LGBM regressor with 3-fold CV and step size of 2.
    scaler              :   Scaler for feature normalization. Default is MinMaxScaler.
    estimator           :   Estimator for the regression task. Default is a LGBM regressor.
    target_transformer  :   Transformer (or scaler) for the target variable. If set to 'ignore', no transformation is applied. Default is MirrorLogNormScaler. Transformer must implement fit, transform, and inverse_transform methods.
    '''

    def __init__(self, rfecv_kwargs:dict=None, scaler=None, estimator=None, target_transformer=None):

        self.rfecv_kwargs = rfecv_kwargs
        self.scaler = scaler
        self.estimator = estimator
        self.target_transformer = target_transformer

        rfecv_kwargs = self.rfecv_kwargs or {'estimator':lgb.LGBMRegressor(verbose=-1), 'cv':3, 'step':2}
        scaler_ = self.scaler or MinMaxScaler()
        estimator_ = self.estimator or lgb.LGBMRegressor(verbose=-1)
        target_transformer_ = self.target_transformer or MirrorLogNormScaler()

        if self.rfecv_kwargs in (None, False, 'ignore', 'skip'):
            feature_step = ('feature_elimination', 'passthrough')
        else:
            feature_step = ('feature_elimination', RFECV(**self.rfecv_kwargs))

        pipe = Pipeline([
            ('feature_elimination', RFECV(**rfecv_kwargs)),
            ('normalization', scaler_),
            ('estimation', estimator_)
        ])

        if self.target_transformer != 'ignore':
            ttr_pipe = TransformedTargetRegressor(
                regressor=pipe,
                transformer=target_transformer_,
                check_inverse=False
            )
        else:
            ttr_pipe = None
        self.model = ttr_pipe if ttr_pipe else pipe

    def fit(self, X, y):

        '''
        X   :   Training matrix.
        y   :   Target values.
        '''

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            self.model.fit(X, y)
        return self

    def predict(self, X):

        '''
        X   :   Samples.
        '''

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            return self.model.predict(X)

def willmotts_index(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    wi = 1 - (np.sum((y_true-y_pred)**2) / np.sum((np.abs(y_pred-np.mean(y_true))+(np.abs(y_true-np.mean(y_pred))))**2))
    return wi

def nash_sutcliffe_efficiency(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ns = 1 - np.sum((y_true-y_pred)**2) / np.sum((y_true-np.mean(y_true))**2)
    return ns

def legates_mccabes_index(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    lm = 1 - np.sum(np.abs(y_pred-y_true)) / np.sum(np.abs(y_true-np.mean(y_true)))
    return lm

def kling_gupta_efficiency(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    cv_true = np.std(y_true) / np.mean(y_true)
    cv_pred = np.std(y_pred) / np.mean(y_pred)
    r = np.sum((y_true - y_true.mean()) * (y_pred - y_pred.mean())) / np.sqrt(np.sum((y_true - y_true.mean())**2) * np.sum((y_pred - y_pred.mean())**2))
    kge = 1 - np.sqrt((r-1)**2 + (np.mean(y_pred)/np.mean(y_true) - 1)**2 + (cv_pred/cv_true)**2)
    return kge

def normalized_root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    nrmse = root_mean_squared_error(y_true=y_true, y_pred=y_pred) / np.mean(y_true)
    return nrmse

def relative_mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rmae = mean_absolute_error(y_true=y_true, y_pred=y_pred) / np.mean(y_true)
    return rmae

def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    smape = (1/len(y_true)) * np.sum(np.abs(y_true-y_pred) / ((np.abs(y_true) + np.abs(y_pred))/2))
    return smape

def theils_inequality_coefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    numerator = np.sqrt((1/n) * (np.sum(y_pred-y_true)**2))
    denominator = np.sqrt((1/n) * np.sum(y_true**2)) + np.sqrt((1/n) * np.sum(y_pred**2))
    tic = numerator / denominator
    return tic

def absolute_percentage_bias(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    apb = np.abs(np.sum(y_true-y_pred) / np.sum(y_true))
    return apb

def evaluate_model(model, X, y_true) -> pd.DataFrame:
    '''
    model   :   A pre-trained model object.
    X       :   Feature matrix. Should be in the format your model object requires.
    y_true  :   True target array for prediction evaluation.
    '''

    if hasattr(model, 'predict'):
        y_pred = model.predict(X)
    elif callable(model):
        y_pred = model(X)
    else:
        raise TypeError('Model must be callable or have a .predict() method.')

    abbr = ['rmse', 'nrmse']
    prop = ['bias', 'bias']
    metrics = [root_mean_squared_error, normalized_root_mean_squared_error]

    results = [metric(y_true=y_true, y_pred=y_pred) for metric in metrics]

    df = pd.DataFrame(
        data=list(zip(prop, abbr, results)),
        columns=['property', 'metric', 'score']
    )

    return df

## Data Collection

### Weather Data

In [6]:
# coordinates = [
#     (52.5200, 13.4050),  # Berlin
#     (53.5488, 9.9872),   # Hamburg
#     (48.1351, 11.5820),  # Munich
#     (50.9375, 6.9603)    # Cologne
# ]
# weather_variables = [
#     'precipitation',
#     'cloud_cover',
#     'sunshine',
#     'temperature',
#     'relative_humidity'
# ]

In [7]:
# %%time

# url = 'https://api.brightsky.dev/weather'
# parameters = {
#     'date':'2024-01-01T01:00:00Z',
#     'last_date':'2025-01-01T00:00:00Z'
# }
# concat_list = []
# for lat, lon in coordinates:
#     parameters['lat'] = lat
#     parameters['lon'] = lon
#     data = requests.get(url, parameters).json()
#     temp = pd.DataFrame(data['weather'])[['timestamp'] + weather_variables]
#     concat_list.append(temp)
# weather = pd.concat(concat_list)
# weather['datetime'] = pd.to_datetime(weather['timestamp'], format='ISO8601', utc=True) + pd.Timedelta(hours=-1)
# weather = weather\
#     .drop(columns='timestamp')\
#     .groupby('datetime', as_index=False)\
#     .agg('mean')
# print(weather.shape)
# weather.head()

### Day Ahead Price Data

In [8]:
# %%time

# url = 'https://api.energy-charts.info/price'
# parameters = {
#     'start':'2024-01-01T00:00:00Z',
#     'end':'2024-12-31T23:00:00Z'
# }
# data = requests.get(url, params=parameters).json()
# data.pop('license_info'); data.pop('unit'); data.pop('deprecated')
# price = pd.DataFrame(data)
# price['datetime'] = pd.to_datetime(price['unix_seconds'], unit='s', utc=True)
# price = price.drop(columns='unix_seconds')
# print(price.shape)
# price.head()

### Production Data

In [9]:
# %%time

# url = 'https://api.energy-charts.info/public_power'
# parameters = {
#     'start':'2024-01-01T01:00:00Z',
#     'end':'2025-01-01T00:00:00Z'
# }
# data = requests.get(url, params=parameters).json()
# public_power = pd.DataFrame()
# public_power['datetime'] = pd.to_datetime(data['unix_seconds'], unit='s', utc=True) + pd.Timedelta(hours=-1)
# for production_type_data in data['production_types']:
#     col = production_type_data['name'].lower().replace(' ', '_').replace('-', '_')
#     public_power[col] = production_type_data['data']
# public_power = public_power[
#     public_power['datetime'].dt.minute == 0
# ].drop(columns=['hydro_pumped_storage_consumption', 'cross_border_electricity_trading'])
# print(public_power.shape)
# public_power.head()

### Cross-Border Electricity Trading Data

In [10]:
# %%time

# url = 'https://api.energy-charts.info/cbet'
# parameters = {
#     'start':'2024-01-01T01:00:00Z',
#     'end':'2025-01-01T00:00:00Z'
# }
# data = requests.get(url, parameters).json()
# cbet = pd.DataFrame()
# cbet['datetime'] = pd.to_datetime(data['unix_seconds'], unit='s', utc=True) + pd.Timedelta(hours=-1)
# for country_data in data['countries']:
#     col = country_data['name'].lower().replace(' ', '_').replace('-', '_') + '_cbet'
#     cbet[col] = country_data['data']
# cbet = cbet[
#     cbet['datetime'].dt.minute == 0
# ]
# print(cbet.shape)
# cbet.head()

### Data Concatenation

In [11]:
# df = reduce(
#     lambda l, r : l.merge(r, on='datetime', how='outer'),
#     [weather, price, public_power, cbet]
# ).sort_values(by='datetime', ascending=True)
# df.to_csv('dataset.csv', index=False)
# print(df.shape)
# df.head()

In [12]:
# print(df['datetime'].min(), df['datetime'].max())

# df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
# df = df.set_index('datetime')

In [13]:
# # Variables selection
# vars = [
#     'precipitation',
#     'cloud_cover',
#     'sunshine',
#     'temperature',
#     'relative_humidity',
#     'load',
#     'price',
#     'sum_cbet'
# ]
# X_df = df[vars].copy()

# # Incoporating lag variables
# X_lag = pd.concat(
#     [X_df[col].shift(lag).rename(f'{col}_lag{lag}')
#     for col in X_df.columns
#     for lag in range(1, 25)],
#     axis = 1
# ).dropna()

# y = df.loc[X_lag.index, 'price']

# target_transformer = MirrorLogNormScaler(
#     mirrorlog_kwargs={'c':1/3},
#     normalizer=MinMaxScaler()
# )

# model_fit = ProcessingPipeline(
#     rfecv_kwargs = 'skip',
#     scaler=MinMaxScaler(),
#     estimator= xgb.XGBRegressor(
#         n_estimators=500,
#         learning_rate=0.05,
#         max_depth=6,
#         subsample=0.9,
#         colsample_bytree=0.9,
#         reg_lambda=1.0,
#         tree_method="hist",
#         random_state=42
#     ),
#     target_transformer=target_transformer
# )


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
# Replace 'your_file_path.csv' with the actual path to your CSV file in Google Drive
csv_file_path = '/content/drive/MyDrive/MURES/rfe_dataset_2019_2025.csv'
df_rfe = pd.read_csv(csv_file_path)

# Setting Index
df_rfe['datetime'] = pd.to_datetime(df_rfe['datetime'], utc=True)
df_rfe = df_rfe.set_index('datetime')

y = df_rfe['price']
X_lag = df_rfe.drop(columns=['price'])

###################### Grid Search Tuning

# model_tpl = ProcessingPipeline(
#     rfecv_kwargs=None,
#     scaler= None,
#     estimator=xgb.XGBRegressor(
#         n_estimators=150,
#         tree_method='hist',
#         random_state=42,
#         n_jobs=-1,
#         objective = 'reg:squarederror'
#     ),
#     target_transformer=None
# )

# TransformedTargetRegressor y to log
# wrapped = TransformedTargetRegressor(
#     regressor=model_tpl,
#     transformer=FunctionTransformer(np.log1p, np.expm1)
# )

nrmse_scorer = make_scorer(normalized_root_mean_squared_error, greater_is_better=False)


In [16]:
# df_xy = X_lag.copy()
# df_xy["__y__"] = y
# df_xy = df_xy.dropna()

# # 2) 다시 분리
# y_clean = df_xy["__y__"]
# X_clean = df_xy.drop(columns="__y__")

# # Split the Data into Train and Test
# X_train_val, X_test, y_train_val, y_test = train_test_split(X_clean, y_clean, test_size=0.1, shuffle=False)

# train_size = int(len(X_train_val) * (0.9*0.8))  # 0.9 because train needs to be 80% of train/val dataset & validation shoudld be 90% of total dataset
# step_size = 24*7*20  # every fold is 20 weeks in length
# n_splits = (len(X_train_val) - train_size) // step_size
# tscv = TimeSeriesSplit(n_splits=n_splits, max_train_size=train_size)

# # GridSearchCV
# search = GridSearchCV(
#     estimator=wrapped,
#     param_grid={
#         "regressor__estimator__max_depth": [4, 6, 8],
#         "regressor__estimator__learning_rate": [0.03, 0.05, 0.07],
#         "regressor__estimator__subsample": [0.7, 0.85, 1.0],
#         "regressor__estimator__min_child_weight": [1, 5, 10],
#         "regressor__estimator__colsample_bytree": [0.6, 0.8, 1.0],
#         "regressor__estimator__reg_alpha": [0, 1e-3, 1e-2],
#         "regressor__estimator__reg_lambda": [1, 5, 10],
#     },
#     scoring=nrmse_scorer,
#     cv=tscv,
#     n_jobs=-1,
#     verbose=2
# )

# # Fitting model
# search.fit(X_train_val, y_train_val)

# # Evaluating model
# evaluate_model(search, X_test, y_test)

In [17]:
import inspect

def _fit_supports(argname, estimator):
    sig = inspect.signature(estimator.fit)
    return argname in sig.parameters

def cv_with_early_stopping(X, y, tscv, params, nrmse_scorer):
    scores = []
    for tr, va in tscv.split(X, y):
        Xtr, Xva = X.iloc[tr], X.iloc[va]
        ytr, yva = y.iloc[tr], y.iloc[va]

        model = XGBRegressor(
            n_estimators=1000,
            tree_method='hist',
            random_state=42,
            n_jobs=-1,
            objective='reg:squarederror',
            eval_metric='rmse',
            **params
        )

        fit_kwargs = dict(
            X=Xtr, y=ytr,
            eval_set=[(Xva, yva)],
            verbose=False
        )

        # 1) callbacks → Using EarlyStopping
        if _fit_supports("callbacks", model):
            from xgboost.callback import EarlyStopping
            fit_kwargs["callbacks"] = [EarlyStopping(rounds=100, save_best=True, maximize=False)]
        # 2) early_stopping_rounds
        elif _fit_supports("early_stopping_rounds", model):
            fit_kwargs["early_stopping_rounds"] = 100

        model.fit(**fit_kwargs)

        score = nrmse_scorer(model, Xva, yva) if callable(getattr(nrmse_scorer, '__call__', None)) else float('nan')
        scores.append(score)

    return float(np.nanmean(scores))

# Removing NA values
df_xy = X_lag.copy()
df_xy["__y__"] = y
df_xy = df_xy.dropna()

y_clean = df_xy["__y__"]
X_clean = df_xy.drop(columns="__y__")

# Split the Data into Train and Test
X_train_val, X_test, y_train_val, y_test = train_test_split(X_clean, y_clean, test_size=0.1, shuffle=False)

train_size = int(len(X_train_val) * (0.9*0.8))  # 0.9 because train needs to be 80% of train/val dataset & validation shoudld be 90% of total dataset
step_size = 24*7*20  # every fold is 20 weeks in length
n_splits = (len(X_train_val) - train_size) // step_size
tscv = TimeSeriesSplit(n_splits=n_splits, max_train_size=train_size)

# Param_grid
param_grid_es = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "min_child_weight": [1, 5],
    "reg_alpha": [0, 1e-3],
    "reg_lambda": [1, 5],
}

best_score, best_params = -np.inf, None
for params in ParameterGrid(param_grid_es):
    s = cv_with_early_stopping(X_train_val, y_train_val, tscv, params, nrmse_scorer)
    if s > best_score:
        best_score, best_params = s, params
        print(f"[cv] score={s:.6f} params={params}")

print("BEST:", best_score, best_params)

# Final Training
final_model = XGBRegressor(
    n_estimators=1000,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror',
    eval_metric='rmse',
    **best_params
)

split = int(len(X_train_val)*0.9)
fit_kwargs = dict(
    X=X_train_val.iloc[:split], y=y_train_val.iloc[:split],
    eval_set=[(X_train_val.iloc[split:], y_train_val.iloc[split:])],
    verbose=False
)
if _fit_supports("callbacks", final_model):
    from xgboost.callback import EarlyStopping
    fit_kwargs["callbacks"] = [EarlyStopping(rounds=100, save_best=True, maximize=False)]
elif _fit_supports("early_stopping_rounds", final_model):
    fit_kwargs["early_stopping_rounds"] = 100

final_model.fit(**fit_kwargs)

class SearchLike:
    best_estimator_ = final_model
    best_params_    = best_params

search_es = SearchLike()


[cv] score=-0.568374 params={'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 0.8}
[cv] score=-0.558254 params={'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 1.0}
[cv] score=-0.554611 params={'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'reg_alpha': 0, 'reg_lambda': 5, 'subsample': 0.8}
[cv] score=-0.553330 params={'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 5, 'reg_alpha': 0, 'reg_lambda': 5, 'subsample': 1.0}
[cv] score=-0.550670 params={'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 5, 'reg_alpha': 0.001, 'reg_lambda': 5, 'subsample': 1.0}
BEST: -0.5506702264597113 {'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 5, 'reg_alpha': 0.001, 'reg_lambda': 5, 'subsample

In [18]:
evaluate_model(final_model, X_test, y_test)

,property,metric,score
0,bias,rmse,44.999852
1,bias,nrmse,0.516434
